# Fraud neural-network training

This notebook is self-contained and intended for the course JupyterHub.
It reads the shared dataset from `/data/mlproject22/transactions.csv.zip`
and saves the selected model to the project's `models` directory.



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.nn import functional as F

SEED = 42
torch.manual_seed(SEED)

current_dir = Path.cwd()
root_dir = current_dir.parent if current_dir.name.lower() == "code" else current_dir

DATA_FILE = Path("/data/mlproject22/transactions.csv.zip")
MODEL_DIR = root_dir / "models"
MODEL_FILE = MODEL_DIR / "best_validation_model.pt"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Shared training data not found at {DATA_FILE}. "
        "Check the course's JupyterHub dataset path."
    )

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", current_dir)
print("Training data:", DATA_FILE)
print("Model output:", MODEL_FILE)



Working directory: /home/csav7391/ml_project-financial_transactions/Code
Training data: /data/mlproject22/transactions.csv.zip
Model output: /home/csav7391/ml_project-financial_transactions/models/best_validation_model.pt


## Load and split the data

Hyperparameters are selected using the validation set. The test set is used
only once after the best configuration has been selected.



In [2]:
def load_data(file, target="Class", test_size=0.2, validation_size=0.2):
    frame = pd.read_csv(file)
    X = frame.drop(columns=target)
    y = frame[target].to_numpy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=SEED,
    )
    X_subtrain, X_validation, y_subtrain, y_validation = train_test_split(
        X_train,
        y_train,
        test_size=validation_size,
        stratify=y_train,
        random_state=SEED,
    )

    search_scaler = StandardScaler().fit(X_subtrain)
    X_subtrain = search_scaler.transform(X_subtrain).astype("float32")
    X_validation = search_scaler.transform(X_validation).astype("float32")

    final_scaler = StandardScaler().fit(X_train)
    X_train = final_scaler.transform(X_train).astype("float32")
    X_test = final_scaler.transform(X_test).astype("float32")

    return {
        "feature_columns": X.columns.tolist(),
        "X_subtrain": X_subtrain,
        "X_validation": X_validation,
        "y_subtrain": y_subtrain,
        "y_validation": y_validation,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "scaler": final_scaler,
    }


data = load_data(DATA_FILE)
print("Training rows:", len(data["y_train"]))
print("Validation rows:", len(data["y_validation"]))
print("Test rows:", len(data["y_test"]))
print("Features:", len(data["feature_columns"]))



Training rows: 182276
Validation rows: 36456
Test rows: 45569
Features: 30


## Neural network and evaluation



In [3]:
class Network(nn.Module):
    def __init__(self, input_size, hidden_sizes):
        super().__init__()
        sizes = [input_size, *hidden_sizes, 1]
        self.layers = nn.ModuleList(
            nn.Linear(input_features, output_features)
            for input_features, output_features in zip(sizes, sizes[1:])
        )

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = torch.sigmoid(layer(x))
        return self.layers[-1](x)


def train(
    X,
    y,
    hidden_sizes=(4,),
    epochs=25,
    learning_rate=0.01,
    pos_weight=100.0,
    seed=SEED,
):
    torch.manual_seed(seed)
    X_tensor = torch.as_tensor(X, dtype=torch.float32)
    y_tensor = torch.as_tensor(y, dtype=torch.float32).reshape(-1, 1)

    model = Network(X_tensor.shape[1], hidden_sizes)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    model.train()
    for _ in range(epochs):
        logits = model(X_tensor)
        loss = F.binary_cross_entropy_with_logits(
            logits,
            y_tensor,
            pos_weight=logits.new_tensor([pos_weight]),
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model


def evaluate(model, X, y, threshold=0.5):
    model.eval()
    with torch.no_grad():
        logits = model(torch.as_tensor(X, dtype=torch.float32))
        scores = torch.sigmoid(logits).flatten().cpu().numpy()

    predictions = (scores >= threshold).astype(int)
    metrics = {
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, scores),
        "pr_auc": average_precision_score(y, scores),
        "confusion_matrix": confusion_matrix(y, predictions).tolist(),
    }
    return scores, metrics



## Select a configuration

The configuration with the highest validation ROC-AUC is selected.



In [4]:
CONFIGS = [
    {"hidden_sizes": (), "epochs": 50, "learning_rate": 0.01, "pos_weight": 100},
    {"hidden_sizes": (16,), "epochs": 25, "learning_rate": 0.01, "pos_weight": 100},
    {"hidden_sizes": (32, 16), "epochs": 25, "learning_rate": 0.01, "pos_weight": 100},
    {"hidden_sizes": (64, 32), "epochs": 25, "learning_rate": 0.01, "pos_weight": 100},
    {"hidden_sizes": (96, 48), "epochs": 25, "learning_rate": 0.01, "pos_weight": 100},
    {"hidden_sizes": (32, 16), "epochs": 50, "learning_rate": 0.005, "pos_weight": 160},
    {"hidden_sizes": (64, 32), "epochs": 50, "learning_rate": 0.005, "pos_weight": 160},
    {"hidden_sizes": (128, 64, 32), "epochs": 50, "learning_rate": 0.005, "pos_weight": 160},
]

best_config = None
best_metrics = None
best_model = None

for config in CONFIGS:
    candidate = train(data["X_subtrain"], data["y_subtrain"], **config)
    _, metrics = evaluate(candidate, data["X_validation"], data["y_validation"])
    print(
        f"{config} -> validation ROC-AUC={metrics['roc_auc']:.4f}, "
        f"PR-AUC={metrics['pr_auc']:.4f}"
    )

    if best_metrics is None or metrics["roc_auc"] > best_metrics["roc_auc"]:
        best_config = config
        best_metrics = metrics
        best_model = candidate

print("\nSelected configuration:", best_config)
print("Validation metrics:", best_metrics)



{'hidden_sizes': (), 'epochs': 50, 'learning_rate': 0.01, 'pos_weight': 100} -> validation ROC-AUC=0.9330, PR-AUC=0.6964
{'hidden_sizes': (16,), 'epochs': 25, 'learning_rate': 0.01, 'pos_weight': 100} -> validation ROC-AUC=0.9577, PR-AUC=0.6265
{'hidden_sizes': (32, 16), 'epochs': 25, 'learning_rate': 0.01, 'pos_weight': 100} -> validation ROC-AUC=0.9609, PR-AUC=0.7044
{'hidden_sizes': (64, 32), 'epochs': 25, 'learning_rate': 0.01, 'pos_weight': 100} -> validation ROC-AUC=0.9592, PR-AUC=0.6698
{'hidden_sizes': (96, 48), 'epochs': 25, 'learning_rate': 0.01, 'pos_weight': 100} -> validation ROC-AUC=0.9587, PR-AUC=0.6865
{'hidden_sizes': (32, 16), 'epochs': 50, 'learning_rate': 0.005, 'pos_weight': 160} -> validation ROC-AUC=0.9652, PR-AUC=0.6757
{'hidden_sizes': (64, 32), 'epochs': 50, 'learning_rate': 0.005, 'pos_weight': 160} -> validation ROC-AUC=0.9671, PR-AUC=0.6972
{'hidden_sizes': (128, 64, 32), 'epochs': 50, 'learning_rate': 0.005, 'pos_weight': 160} -> validation ROC-AUC=0.9689,

## Train the final model and save it

The final model is retrained on the complete training split. Its weights,
feature order, and scaler are stored together for leaderboard prediction.



In [5]:
final_model = train(data["X_train"], data["y_train"], **best_config)
_, test_metrics = evaluate(final_model, data["X_test"], data["y_test"])

artifact = {
    "model_state_dict": final_model.state_dict(),
    "input_size": data["X_train"].shape[1],
    "hidden_sizes": best_config["hidden_sizes"],
    "training_config": best_config,
    "feature_columns": data["feature_columns"],
    "scaler_mean": data["scaler"].mean_,
    "scaler_scale": data["scaler"].scale_,
    "threshold": 0.5,
    "validation_metrics": best_metrics,
    "test_metrics": test_metrics,
}
torch.save(artifact, MODEL_FILE)

print("Test metrics:", test_metrics)
print("Saved model:", MODEL_FILE.resolve())
print("Model exists:", MODEL_FILE.exists())



Test metrics: {'precision': 0.09263959390862944, 'recall': 0.9240506329113924, 'f1': 0.16839677047289503, 'roc_auc': 0.9844453225218507, 'pr_auc': 0.7957419262961295, 'confusion_matrix': [[44775, 715], [6, 73]]}
Saved model: /home/csav7391/ml_project-financial_transactions/models/best_validation_model.pt
Model exists: True


## Verify that the checkpoint can be loaded



In [6]:
saved = torch.load(MODEL_FILE, map_location="cpu", weights_only=False)
loaded_model = Network(saved["input_size"], tuple(saved["hidden_sizes"]))
loaded_model.load_state_dict(saved["model_state_dict"])
loaded_model.eval()

print("Checkpoint loaded successfully.")
print("Architecture:", saved["hidden_sizes"])
print("Saved features:", len(saved["feature_columns"]))


Checkpoint loaded successfully.
Architecture: (128, 64, 32)
Saved features: 30
